<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# Intel VIP Verification Components

This tutorial connects Intel VIP packet objects to a pyuvm verification environment. Object construction and reference-model code are ordinary Python; agents, drivers, monitors, and scoreboards run inside a cocotb simulation.

## 1. Verification architecture

```text
VIPSequence → VIPDriver → Avalon-ST source → DUT → Avalon-ST sink
                            │                    │
                       source monitor       sink monitor
                            │                    │
                            ▼                    ▼
                    BaseVIPPredictor → BaseVIPScoreboard
```

The packet codec remains independent of simulation. `VIPDriver` serializes packet objects, while `VIPMonitor` reconstructs them and publishes them through a pyuvm analysis port.

In [1]:
from fpga_verification.protocols.avalon_st.intel_video import (
    IntelVIPFrameCodec,
    VIPControlPacket,
    VIPUserPacket,
)
from fpga_verification.sim.agents import (
    VIPAgent,
    VIPDriver,
    VIPItem,
    VIPMonitor,
    VIPSequence,
)
from fpga_verification.sim.buses import AvalonSTBus
from fpga_verification.sim.models import BaseVIPPredictor, PacketExpectation
from fpga_verification.sim.scoreboards import AnalysisImp, BaseVIPScoreboard
from fpga_verification.video import FrameSize, ImageGenerator, VideoFormat

## 2. VIPItem and VIPSequence

`VIPItem` is a pyuvm sequence item containing exactly one complete VIP packet. `VIPSequence.from_packets()` converts an ordered packet list into items. Its `body()` sends every item through a sequencer.

A frame is normally sent as separate user, control, and video items so that packet boundaries remain visible to the driver and monitor.

In [2]:
fmt = VideoFormat(
    bits_per_color=8,
    number_of_color_planes=3,
    pixels_in_parallel=1,
)
size = FrameSize(width=4, height=2)
frame = ImageGenerator(fmt).horizontal_ramp(size)
packets = IntelVIPFrameCodec(fmt).frame_to_packets(
    frame,
    size,
    user_packets=[VIPUserPacket(1, [0xA, 0xB])],
)

item = VIPItem.from_packet(packets[0])
sequence = VIPSequence.from_packets(packets, name="input_frame")

print(type(item.to_vip_packet()).__name__)
print([type(sequence_item.packet).__name__ for sequence_item in sequence.items])

VIPUserPacket
['VIPUserPacket', 'VIPControlPacket', 'VIPVideoPacket']


## 3. VIPDriver and VIPMonitor

`VIPDriver` receives `VIPItem` objects, calls `packet.to_symbols()`, and sends one `AvalonSTFrame` per packet. It is normally created by an active `VIPAgent`.

`VIPMonitor` receives complete Avalon-ST frames, decodes them with `vip_packet_from_symbols()`, checks ordering with `VIPProtocolChecker`, and publishes packet objects. With `drive_ready=False` it observes passively; with `drive_ready=True` it acts as a sink and controls backpressure.

In [3]:
def make_passive_monitor(parent, dut, fmt):
    return VIPMonitor(
        "vip_tap",
        parent,
        bus=AvalonSTBus.from_prefix(dut, "vip_out"),
        clock=dut.clk,
        reset=dut.reset,
        fmt=fmt,
        drive_ready=False,
        packet_logging=True,
    )

## 4. VIPAgent

`VIPAgent` assembles the source driver, sequencer, source monitor, and/or sink monitor. Provide either bus or both buses:

- `source_bus` is driven toward the DUT and also monitored;
- `sink_bus` is observed after the DUT and, in an active agent, drives `ready`;
- a passive agent only observes existing traffic;
- `randomize=True` introduces source pauses and sink backpressure;
- `source_analysis_port` and `sink_analysis_port` publish decoded packets.

In [4]:
def make_active_agent(parent, dut, fmt):
    return VIPAgent(
        "vip_agent",
        parent,
        clock=dut.clk,
        reset=dut.reset,
        source_bus=AvalonSTBus.from_prefix(dut, "vip_in"),
        sink_bus=AvalonSTBus.from_prefix(dut, "vip_out"),
        source_fmt=fmt,
        sink_fmt=fmt,
        randomize=True,
        packet_logging=True,
    )

# During a pyuvm run phase:
# await sequence.start(agent.sequencer)

## 5. PacketExpectation and BaseVIPPredictor

A predictor consumes input packets and produces one `PacketExpectation` for each expected output packet. An expectation contains the reference packet, whether it should be compared, an optional skip reason, and pixel tolerance.

`BaseVIPPredictor` handles control packets, converts video packets into numpy frames, calls a frame model, and converts the expected frame back into a video packet. Subclasses normally implement `get_tolerance()` and `_process_user_packet()`, and may override size mapping or supported resolutions.

In [5]:
class IdentityModel:
    def process(self, frame):
        return frame.copy()


class IdentityPredictor(BaseVIPPredictor):
    def get_tolerance(self):
        return 0

    def _process_user_packet(self, packet):
        return PacketExpectation(packet=packet)


codec = IntelVIPFrameCodec(fmt)
predictor = IdentityPredictor(IdentityModel(), codec, codec)
expectations = [predictor.process_packet(packet) for packet in packets]
print([type(item.packet).__name__ for item in expectations])

['VIPUserPacket', 'VIPControlPacket', 'VIPVideoPacket']


## 6. AnalysisImp and BaseVIPScoreboard

`AnalysisImp` adapts a pyuvm analysis export to a Python callback. `BaseVIPScoreboard` uses two of them:

- `data_in_export` sends input packets through an optional predictor;
- `data_out_export` compares output packets against queued expectations.

Control packets are compared structurally. Video packets are decoded and compared as numpy frames with the expectation tolerance. `wait_frame_checked()` lets a test wait until an output frame has been checked and re-raises recorded failures.

In [6]:
def connect_vip_environment(env, agent, scoreboard, predictor):
    scoreboard.predictor = predictor
    agent.source_analysis_port.connect(scoreboard.data_in_export)
    agent.sink_analysis_port.connect(scoreboard.data_out_export)


# Typical pyuvm build phase:
# env.agent = make_active_agent(env, cocotb.top, fmt)
# env.scoreboard = BaseVIPScoreboard(
#     "scoreboard", env, source_fmt=fmt, sink_fmt=fmt
# )
# connect_vip_environment(env, env.agent, env.scoreboard, predictor)

## 7. Lifecycle and cleanup

Use `agent.set_packet_logging()` to change packet summaries and `agent.set_randomize()` to enable or disable pauses after construction. `cancel_bfms()` stops background bus tasks; `clear_bfms()` clears queued traffic and resets monitor protocol state. Reset automatically clears the active VIP control resolution in each monitor.